## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:5'

%load_ext autoreload
%autoreload 2

## Data generation

In [22]:
from datasets import Spiral

n, d = 10000, 64                                
true_rho = 0.7                                   # < higher rho, higher MI
v = 3.14/2

dataset = Spiral.Spiral(rho=true_rho, dim=d, v=v)
X, Y = dataset.sample(n=10000)
X, Y = X.to(device).clone().detach(), Y.to(device).clone().detach()


print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", dataset.MI())

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.773512852220248


## MI estimation

In [24]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [25]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.4923958778381348 loss val= 1.251880168914795 best val loss= 1.251880168914795 best t= 0
finished: t= 63 loss= 0.44867244362831116 loss val= 0.6082248687744141 best val loss= 0.4865320026874542 best t= 61
finished: t= 126 loss= 0.7715884447097778 loss val= 0.528569221496582 best val loss= 0.4865320026874542 best t= 61
finished: t= 189 loss= 0.5119436979293823 loss val= 0.5999642014503479 best val loss= 0.4865320026874542 best t= 61
finished: t= 252 loss= 0.48699697852134705 loss val= 0.7321317195892334 best val loss= 0.4865320026874542 best t= 61
finished: t= 315 loss= 0.47503653168678284 loss val= 0.4795566201210022 best val loss= 0.4745996594429016 best t= 255
finished: t= 378 loss= 0.5028965473175049 loss val= 0.7260851263999939 best val loss= 0.4745996594429016 best t= 255
finished: t= 441 loss= 0.7535560727119446 loss val= 0.6042710542678833 best val loss= 0.46760618686676025 best t= 403
finished: t= 504 loss= 0.49322575330734253 loss val= 0.6023712754249573 

In [29]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.022210951894521713 loss val= 0.019252339377999306 best val loss= 0.019252339377999306 best t= 0
finished: t= 63 loss= -5.293997764587402 loss val= -5.727870941162109 best val loss= -5.894803524017334 best t= 55
finished: t= 126 loss= -5.650267601013184 loss val= -5.445831775665283 best val loss= -6.1947526931762695 best t= 123
finished: t= 189 loss= -5.814635276794434 loss val= -6.057534694671631 best val loss= -6.1947526931762695 best t= 123
finished: t= 252 loss= -5.285787582397461 loss val= -5.565487861633301 best val loss= -6.1947526931762695 best t= 123
finished: t= 315 loss= -6.196941375732422 loss val= -6.073263168334961 best val loss= -6.2884016036987305 best t= 292
finished: t= 378 loss= -6.2393903732299805 loss val= -6.173935890197754 best val loss= -6.2884016036987305 best t= 292
finished: t= 441 loss= -7.186225891113281 loss val= -6.166661262512207 best val loss= -6.2884016036987305 best t= 292


true MI: 10.773512852220248
est MI: 7.446191787719727


In [30]:
## Vector copula-based MI estimation, ver 1
from estimators.VCE import VCE

hyperparams.nde_type = 'VGC'

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

K components= 5 copula transform= True
nde type: VGC
finished: t= 0 loss= 325.6348876953125 loss val= 329.4525451660156 best val loss= 329.4525451660156 best t= 0


finished: t= 51 loss= 45.31538772583008 loss val= 45.60783767700195 best val loss= 45.4883918762207 best t= 41
finished: t= 102 loss= 44.064857482910156 loss val= 48.317481994628906 best val loss= 45.4883918762207 best t= 41
finished: t= 153 loss= 40.977813720703125 loss val= 60.01613235473633 best val loss= 45.4883918762207 best t= 41
finished: t= 204 loss= 37.875099182128906 loss val= 71.69425201416016 best val loss= 45.4883918762207 best t= 41


finished: t= 0 loss= 360.2887268066406 loss val= 359.14599609375 best val loss= 359.14599609375 best t= 0
finished: t= 51 loss= 45.54460906982422 loss val= 45.69029235839844 best val loss= 45.563011169433594 best t= 27
finished: t= 102 loss= 43.80146408081055 loss val= 48.076995849609375 best val loss= 45.563011169433594 best t= 27
finished: t= 153 loss= 41.45099639892578 loss val= 58.53313446044922 best val loss= 45.563011169433594 best t= 27
finished: t= 204 loss= 38.37621307373047 loss val= 72.13621520996094 best val loss= 45.563011169433